# EDA: паттерны совместных просмотров в CoWatch

**Важно:** этот ноутбук по умолчанию читает `data/sample_watch_events.csv` — синтетический демо-датасет (см. `data/README.md`), а не боевую выгрузку из `rooms_db`. Структура полей идентична реальным таблицам, так что весь анализ ниже переносится 1-в-1 на прод-данные — достаточно поменять источник в первой ячейке на подключение к `DATABASE_URL` recommendations-сервиса после того, как ETL (`POST /admin/sync-watch-events`) синхронизировал реальные события.

Цели анализа:
1. Насколько разрежены данные (сколько пользователей вообще подходят под персонализацию, а не только под cold-start popularity fallback).
2. Распределение по жанрам — что вообще смотрят в комнатах.
3. Динамика просмотров во времени.
4. Проверка гипотезы: пользователи кластеризуются по жанровым предпочтениям (обосновывает content-based подход).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

USE_LIVE_DB = False  # переключить на True, когда есть доступ к recommendations_db с реальными данными

if USE_LIVE_DB:
    import os

    from sqlalchemy import create_engine
    engine = create_engine(os.environ["DATABASE_URL"].replace("+asyncpg", ""))
    interactions = pd.read_sql("SELECT * FROM watch_events", engine, parse_dates=["joined_at"])
    content = pd.read_sql("SELECT * FROM content_items", engine)
else:
    interactions = pd.read_csv("../data/sample_watch_events.csv", parse_dates=["joined_at"])
    content = pd.read_csv("../data/sample_content.csv")
    content["genres"] = content["genres"].str.split("|")

df = interactions.merge(content, on="content_id", how="left")
df.head()

## 1. Разреженность: сколько просмотров на пользователя

In [ ]:
watches_per_user = df.groupby("user_id").size().sort_values(ascending=False)
print(f"Пользователей: {watches_per_user.shape[0]}")
print(f"Медиана просмотров на пользователя: {watches_per_user.median()}")
print(f"Пользователей с 1 просмотром (только cold-start fallback): {(watches_per_user == 1).sum()}")
watches_per_user.plot(kind="bar", figsize=(8, 3), title="Просмотров на пользователя")
plt.tight_layout()
plt.show()

## 2. Распределение по жанрам

In [ ]:
genre_counts = df.explode("genres")["genres"].value_counts()
genre_counts.plot(kind="barh", figsize=(7, 5), title="Просмотры по жанрам")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Динамика просмотров во времени (по неделям)

In [ ]:
weekly = df.set_index("joined_at").resample("W").size()
weekly.plot(figsize=(8, 3), marker="o", title="Просмотров в неделю")
plt.tight_layout()
plt.show()

## 4. Жанровые профили пользователей (обоснование content-based подхода)

Если пользователи явно группируются по доминирующему жанру — content-based
рекомендации (TF-IDF на genres+overview, `app/services/recommender.py`) будут
работать разумно даже без данных о поведении других пользователей.

In [ ]:
user_genre = (
    df.explode("genres")
    .pivot_table(index="user_id", columns="genres", values="content_id", aggfunc="count", fill_value=0)
)
user_genre_share = user_genre.div(user_genre.sum(axis=1), axis=0)
user_genre_share.style.background_gradient(cmap="Blues", axis=1)

## Выводы (на демо-данных)

- Данные разрежены: медиана 3 просмотра на пользователя — collaborative filtering на таком объёме не взлетит, content-based — рабочий выбор для MVP.
- Жанровые профили пользователей действительно кластеризуются (sci-fi / romance-drama / horror / arthouse-comedy) — гипотеза для content-based модели подтверждается даже на игрушечном датасете.
- На реальных данных (после первого прогона ETL) этот ноутбук нужно перезапустить с `USE_LIVE_DB = True` и обновить выводы — на демо-данных они иллюстративные, не про реальных пользователей CoWatch.